# Axxmorcito · Piper Fast Voice Fine-Tuning (ES + EN)
Entrena en GPU (T4 gratuita de Google Colab) los dos modelos rápidos de Piper (`axx-morcito-es.onnx` y `axx-morcito-en.onnx`) a partir del paquete `axxmorcito-colab-bundle.zip` generado en tu PC.

**Pasos:**
1. En Colab ve a **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU**.
2. Ejecuta la **Celda 1** y sube `axxmorcito-colab-bundle.zip` cuando te lo pida.
3. Ejecuta la **Celda 2** (entrena español e inglés y descarga automáticamente `axxmorcito-piper-models.zip`).
4. En tu PC ejecuta: `python scripts/voice/install_piper_bundle.py axxmorcito-piper-models.zip`

In [ ]:
# Celda 1: Instalar Piper Trainer + subir axxmorcito-colab-bundle.zip
!apt-get update -qq && apt-get install -y -qq espeak-ng ffmpeg cmake ninja-build
!git clone --depth 1 https://github.com/OHF-Voice/piper1-gpl.git /content/piper1-gpl
%cd /content/piper1-gpl
!pip install -q --upgrade pip wheel setuptools scikit-build cython onnx onnxscript
!pip install -q -e '.[train]'
!./build_monotonic_align.sh && python setup.py build_ext --inplace
import pathlib; p = pathlib.Path('/content/piper1-gpl/src/piper/train/__main__.py'); s = p.read_text(); i = s.find('ModelCheckpoint(\n            monitor="val_mos"'); p.write_text(s[:i] + s[s.find('),', i)+2:] if i != -1 else s)
ep = pathlib.Path('/content/piper1-gpl/src/piper/train/export_onnx.py'); es = ep.read_text(); ep.write_text(es.replace('torch.onnx.export(', 'torch.onnx.export(dynamo=False, ') if 'dynamo=' not in es else es)
!mkdir -p /content/checkpoints /content/data /content/out
!curl -L --fail -o /content/checkpoints/lessac-medium.ckpt \
  'https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_US/lessac/medium/epoch%3D2164-step%3D1355540.ckpt'

from google.colab import files
uploaded = files.upload()  # Sube axxmorcito-colab-bundle.zip
import zipfile, io
for name, raw in uploaded.items():
    with zipfile.ZipFile(io.BytesIO(raw)) as zf:
        zf.extractall('/content/data')
print('Dataset extraído en /content/data')

In [ ]:
# Celda 2: Entrenar Español (es-419) e Inglés (en-us) y exportar ONNX listos para Axx
import json, subprocess, zipfile
from pathlib import Path

EPOCHS = 600  # ~12-15 min por idioma en T4 GPU
JOBS = [
    ('axx-morcito-es', '/content/data/dataset-es', 'es-419', 'Axxmorcito Rápida (ES)', 'es-MX'),
    ('axx-morcito-en', '/content/data/dataset-en', 'en-us', 'Axxmorcito Fast (EN)', 'en-US'),
]

out_root = Path('/content/out')
out_root.mkdir(parents=True, exist_ok=True)

for name, data_dir, espeak_voice, label, lang_code in JOBS:
    work = Path(f'/content/work/{name}')
    work.mkdir(parents=True, exist_ok=True)
    meta_in = Path(data_dir) / 'metadata.csv'
    meta_out = work / 'metadata.csv'
    meta_out.write_text('\n'.join(
        f"{line.split('|')[0]}.wav|{line.split('|', 1)[1]}"
        for line in meta_in.read_text(encoding='utf-8').splitlines() if '|' in line
    ) + '\n', encoding='utf-8')
    subprocess.run([
        'python', '-m', 'piper.train', 'fit',
        '--data.voice_name', name,
        '--data.csv_path', str(meta_out),
        '--data.audio_dir', str(Path(data_dir) / 'wavs'),
        '--model.sample_rate', '22050',
        '--data.espeak_voice', espeak_voice,
        '--data.cache_dir', str(work / 'cache'),
        '--data.config_path', str(work / 'config.json'),
        '--data.batch_size', '16',
        '--data.num_workers', '2',
        '--data.trim_silence', 'false',
        '--data.validation_split', '0.05',
        '--data.num_test_examples', '2',
        '--model.mos_metric', 'none',
        '--trainer.accelerator', 'gpu',
        '--trainer.devices', '1',
        '--trainer.max_epochs', str(EPOCHS),
        '--trainer.check_val_every_n_epoch', '25',
        '--trainer.default_root_dir', str(work),
        '--model.warmstart_ckpt', '/content/checkpoints/lessac-medium.ckpt',
    ], check=True, cwd='/content/piper1-gpl')
    ckpts = sorted(work.glob('lightning_logs/*/checkpoints/*.ckpt'), key=lambda p: p.stat().st_mtime)
    onnx_path = out_root / f'{name}.onnx'
    json_path = out_root / f'{name}.onnx.json'
    subprocess.run([
        'python', '-m', 'piper.train.export_onnx',
        '--checkpoint', str(ckpts[-1]),
        '--output-file', str(onnx_path),
    ], check=True, cwd='/content/piper1-gpl')
    cfg = json.loads((work / 'config.json').read_text(encoding='utf-8'))
    cfg['axx'] = {'label': label, 'language': lang_code}
    json_path.write_text(json.dumps(cfg, ensure_ascii=False, indent=2), encoding='utf-8')

bundle = Path('/content/axxmorcito-piper-models.zip')
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in out_root.glob('axx-morcito-*'):
        zf.write(f, f.name)

from google.colab import files
files.download(str(bundle))